# MODFLOW 6 API B: watching the solver converge

This is notebook B of the API series. Notebook A ([`mf6-api-a-basic`](mf6-api-a-basic.ipynb)) covers the steps of an API run and the outer-iteration loop this notebook opens up.

MODFLOW solves each time step by repeating outer iterations until the largest head change is smaller than the closure criterion. The listing file reports the iterations after the run. Through the API, you can read the head change of every outer iteration while the model runs and plot it as it goes.

## What this notebook covers

By the end of this notebook you will be able to:

- read the largest head change of each outer iteration from the memory of the solution,
- redraw a figure after every time step, so you can watch the model converge as it runs, and
- tell a time step that converges easily from one that is hard to solve.

## Imports

Import FloPy, NumPy, Matplotlib, and `ModflowApi`. `mf6_api_helpers` finds the MODFLOW 6 library and draws the convergence figure, and `BASE_MODEL` from `sv_notebook_helpers` is the folder of the synthetic-valley base model.

In [ ]:
%matplotlib inline
import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import numpy as np
from mf6_api_helpers import find_mf6_libraries, plot_convergence
from modflowapi import ModflowApi
from sv_notebook_helpers import BASE_MODEL

## Find the MODFLOW 6 library

The API runs MODFLOW 6 through its shared library, `libmf6` (`libmf6.so` on Linux, `libmf6.dylib` on macOS, and `libmf6.dll` on Windows), rather than through the `mf6` program. pixi installs both with the class MODFLOW 6. `find_mf6_libraries()` returns the path of each.

In [ ]:
lib_path, mf6_exe = find_mf6_libraries()
print(f"library: {lib_path.name}")
print(f"program: {mf6_exe.name}")

## Start the model from the land surface

The base model starts from its own steady-state heads, so even the steady-state period converges in a few outer iterations. Start it from the land surface instead, so the steady-state solve has further to go: set the starting heads of the Initial Conditions (IC) package, `strt`, to the top of layer 1 in every layer. `np.broadcast_to()` repeats the two-dimensional top for each of the five layers.

In [ ]:
name = "sv"
ws = pl.Path("models/mf6-api-b-convergence")

sim = flopy.mf6.MFSimulation.load(
    sim_ws=BASE_MODEL, exe_name=mf6_exe, write_headers=False, verbosity_level=0
)
sim.set_sim_path(ws)
gwf = sim.get_model(name)
top = gwf.dis.top.array
gwf.ic.strt = np.broadcast_to(top, gwf.dis.botm.array.shape).copy()
sim.write_simulation(silent=True)

## Watch the solver converge

Read `MXITER` with `get_value()`, which returns a copy. Read `HNCG` with `get_value_ptr()`, which returns the array itself: it holds one value per outer iteration, and the solution fills it in as it iterates, so `max_change[kiter]` is the largest head change of outer iteration `kiter` of the current time step. `KPER` and `KSTP` in the Time Discretization package (TDIS) give the stress period and time step, counted from 1.

After each time step, `plot_convergence()` redraws the figure with every outer iteration so far, so it grows as the model runs. The largest head change is plotted on a log scale, because it falls by orders of magnitude within a time step. Each time step gets the same width on the x-axis whatever its number of iterations.

In [ ]:
width = 17.15 / 2.54  # figure width, inches

mf6 = ModflowApi(lib_path, working_directory=ws)
mf6.initialize()
max_iter = int(mf6.get_value(mf6.get_var_address("MXITER", "SLN_1"))[0])
max_change = mf6.get_value_ptr(mf6.get_var_address("HNCG", "SLN_1"))
kper_address = mf6.get_var_address("KPER", "TDIS")
kstp_address = mf6.get_var_address("KSTP", "TDIS")

# one (iteration, |largest head change|, stress period, time step) per
# outer iteration
history = []
niter = 0
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(width, 0.45 * width), layout="constrained")
    while mf6.get_current_time() < mf6.get_end_time():
        mf6.prepare_time_step(mf6.get_time_step())
        kper = int(mf6.get_value(kper_address)[0])
        kstp = int(mf6.get_value(kstp_address)[0])

        mf6.prepare_solve()
        kiter = 0
        converged = False
        while kiter < max_iter and not converged:
            converged = mf6.solve()
            niter += 1
            history.append((niter, abs(float(max_change[kiter])), kper, kstp))
            kiter += 1
        plot_convergence(fig, ax, history)  # redraw once per time step
        if not converged:
            print(f"stress period {kper} did not converge")

        mf6.finalize_solve()
        mf6.finalize_time_step()
    mf6.finalize()
# the figure is already shown; closing it stops it being drawn a second time
plt.close(fig)

In [ ]:
iterations = {}
for _, change, kper, _ in history:
    iterations[kper] = iterations.get(kper, 0) + 1
steady = [h[1] for h in history if h[2] == 1]
print(f"steady state: {iterations[1]} outer iterations")
print(
    f"  largest head change {steady[0]:.2f} m in the first, {steady[-1]:.1e} m in the last"
)
transient = [n for kper, n in iterations.items() if kper > 1]
print(f"transient periods: {min(transient)} to {max(transient)} outer iterations")

**What to look for.** Started from the land surface, the steady-state period takes 32 outer iterations. Its largest head change falls from 5.70 m in the first to 3.3 × 10⁻⁶ m in the 26th, just above the closure criterion of 3.048 × 10⁻⁶ m (`OUTER_DVCLOSE` in the IMS file), then jumps to 1.3 × 10⁻⁴ m and takes 6 more outer iterations to fall to 2.4 × 10⁻⁶ m. Each transient period starts from the heads of the year before and takes 2 to 15 outer iterations. A time step whose head change falls by an order of magnitude or more each outer iteration converges easily. Stress periods 3 and 12 take 10 and 15 outer iterations: in each, the head change stalls between about 10⁻⁶ and 10⁻⁴ m for several iterations before it falls below the closure criterion. Those are the time steps that are hardest to solve, and where to look first if the model converged slowly.

## Recap

You opened the outer-iteration loop of the synthetic-valley base model, read the largest head change of every outer iteration from the memory of the solution while the model ran, and redrew a convergence figure after every time step. The same reading shows which time steps are hard to solve, which is where to look first when a model converges slowly or not at all.